In [117]:
# import libraries 
import re
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

In [118]:
url = "https://jiji.ng/api_web/v1/listing?slug=houses-apartments-for-sale&page=1&webp=true"

response = requests.get(url)

data = response.json()

In [119]:
records = []

for item in data['adverts_list']['adverts']:

    attrs = item.get("attrs", [])

    bedrooms = None
    bathrooms = None
    size = None
    furnishing = None
    
    for attr in attrs:

        name = attr.get("name", "").lower()
        value = attr.get("value", "")

        if "bedroom" in name:
            bedrooms = value

        elif "bathroom" in name:
            bathrooms = value

        elif "property size" in name:
            size = value

        elif "furnishing" in name:
            furnishing = value
    records.append({
        "title": item.get("title"),
        "region": item.get("region"),
        "region_name": item.get("region_name"),
        "region_parent_name": item.get("region_parent_name"),
        "price": item.get("price_title"),
        "bedrooms": bedrooms,
        "bathrooms": bathrooms,
        "property_size": size,
        "furnishing": furnishing,
        "is_boost": item.get("is_boost")
    })

df = pd.DataFrame(records)

df.to_csv("data/jiji_housing_raw.csv", index=False)

df.head(10)

,title,region,region_name,region_parent_name,price,bedrooms,bathrooms,property_size,furnishing,is_boost
0,2bdrm Block of Flats in Ago Palace for sale,"Isolo, Ago Palace",Ago Palace,Isolo,"₦ 270,000,000",2,2,600,Unfurnished,diamond
1,5bdrm Duplex in Ikota for sale,"Lekki, Ikota",Ikota,Lekki,"₦ 650,000,000",5,5,500,Unfurnished,diamond
2,5bdrm House in Ikota for sale,"Lekki, Ikota",Ikota,Lekki,"₦ 600,000,000",5,5,550,Unfurnished,diamond
3,3bdrm Apartment in Ago Palace for sale,"Isolo, Ago Palace",Ago Palace,Isolo,"₦ 90,000,000",3,3,620,Unfurnished,diamond
4,6bdrm Duplex in GRA Phase 1 for sale,"Magodo, GRA Phase 1",GRA Phase 1,Magodo,"₦ 450,000,000",6,6,300,Unfurnished,diamond
5,4bdrm Duplex in Ago Palace for sale,"Isolo, Ago Palace",Ago Palace,Isolo,"₦ 260,000,000",4,4,220,Unfurnished,diamond
6,4bdrm Bungalow in Gberigbe for sale,"Ikorodu, Gberigbe",Gberigbe,Ikorodu,"₦ 22,000,000",4,4,638,Unfurnished,diamond
7,5bdrm Duplex in Abraham Adesanya Estate for sale,"Ajah, Abraham Adesanya Estate",Abraham Adesanya Estate,Ajah,"₦ 260,000,000",5,5,500,Unfurnished,diamond
8,4bdrm Duplex in Chevron for sale,"Lekki, Chevron",Chevron,Lekki,"₦ 200,000,000",4,4,1000,Unfurnished,diamond
9,5bdrm Duplex in Banana Island for sale,"Ikoyi, Banana Island",Banana Island,Ikoyi,"₦ 7,000,000,000",5,5,600,Unfurnished,diamond


In [120]:
#Missing Values

df.isnull().sum()


title                 0
region                0
region_name           0
region_parent_name    0
price                 0
bedrooms              0
bathrooms             0
property_size         0
furnishing            0
is_boost              0
dtype: int64

In [121]:
#Which columns have missing values?
#How will you handle missing property size, bedrooms, bathrooms, or furnishing?
df["bedrooms"] = df["bedrooms"].fillna(0)
df["bathrooms"] = df["bathrooms"].fillna(0)
df["property_size"] = df["property_size"].fillna("Unknown")
df["furnishing"] = df["furnishing"].fillna("Unknown")

df

,title,region,region_name,region_parent_name,price,bedrooms,bathrooms,property_size,furnishing,is_boost
0,2bdrm Block of Flats in Ago Palace for sale,"Isolo, Ago Palace",Ago Palace,Isolo,"₦ 270,000,000",2,2,600,Unfurnished,diamond
1,5bdrm Duplex in Ikota for sale,"Lekki, Ikota",Ikota,Lekki,"₦ 650,000,000",5,5,500,Unfurnished,diamond
2,5bdrm House in Ikota for sale,"Lekki, Ikota",Ikota,Lekki,"₦ 600,000,000",5,5,550,Unfurnished,diamond
3,3bdrm Apartment in Ago Palace for sale,"Isolo, Ago Palace",Ago Palace,Isolo,"₦ 90,000,000",3,3,620,Unfurnished,diamond
4,6bdrm Duplex in GRA Phase 1 for sale,"Magodo, GRA Phase 1",GRA Phase 1,Magodo,"₦ 450,000,000",6,6,300,Unfurnished,diamond
5,4bdrm Duplex in Ago Palace for sale,"Isolo, Ago Palace",Ago Palace,Isolo,"₦ 260,000,000",4,4,220,Unfurnished,diamond
6,4bdrm Bungalow in Gberigbe for sale,"Ikorodu, Gberigbe",Gberigbe,Ikorodu,"₦ 22,000,000",4,4,638,Unfurnished,diamond
7,5bdrm Duplex in Abraham Adesanya Estate for sale,"Ajah, Abraham Adesanya Estate",Abraham Adesanya Estate,Ajah,"₦ 260,000,000",5,5,500,Unfurnished,diamond
8,4bdrm Duplex in Chevron for sale,"Lekki, Chevron",Chevron,Lekki,"₦ 200,000,000",4,4,1000,Unfurnished,diamond
9,5bdrm Duplex in Banana Island for sale,"Ikoyi, Banana Island",Banana Island,Ikoyi,"₦ 7,000,000,000",5,5,600,Unfurnished,diamond


In [122]:
df["price"] = (
    df["price"]
    .astype(str)
    .str.replace("₦", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df["price"] = pd.to_numeric(df["price"], errors="coerce")
df

,title,region,region_name,region_parent_name,price,bedrooms,bathrooms,property_size,furnishing,is_boost
0,2bdrm Block of Flats in Ago Palace for sale,"Isolo, Ago Palace",Ago Palace,Isolo,270000000,2,2,600,Unfurnished,diamond
1,5bdrm Duplex in Ikota for sale,"Lekki, Ikota",Ikota,Lekki,650000000,5,5,500,Unfurnished,diamond
2,5bdrm House in Ikota for sale,"Lekki, Ikota",Ikota,Lekki,600000000,5,5,550,Unfurnished,diamond
3,3bdrm Apartment in Ago Palace for sale,"Isolo, Ago Palace",Ago Palace,Isolo,90000000,3,3,620,Unfurnished,diamond
4,6bdrm Duplex in GRA Phase 1 for sale,"Magodo, GRA Phase 1",GRA Phase 1,Magodo,450000000,6,6,300,Unfurnished,diamond
5,4bdrm Duplex in Ago Palace for sale,"Isolo, Ago Palace",Ago Palace,Isolo,260000000,4,4,220,Unfurnished,diamond
6,4bdrm Bungalow in Gberigbe for sale,"Ikorodu, Gberigbe",Gberigbe,Ikorodu,22000000,4,4,638,Unfurnished,diamond
7,5bdrm Duplex in Abraham Adesanya Estate for sale,"Ajah, Abraham Adesanya Estate",Abraham Adesanya Estate,Ajah,260000000,5,5,500,Unfurnished,diamond
8,4bdrm Duplex in Chevron for sale,"Lekki, Chevron",Chevron,Lekki,200000000,4,4,1000,Unfurnished,diamond
9,5bdrm Duplex in Banana Island for sale,"Ikoyi, Banana Island",Banana Island,Ikoyi,7000000000,5,5,600,Unfurnished,diamond


In [123]:
df["bedrooms"] = pd.to_numeric(df["bedrooms"], errors="coerce")
df["bathrooms"] = pd.to_numeric(df["bathrooms"], errors="coerce")
df

,title,region,region_name,region_parent_name,price,bedrooms,bathrooms,property_size,furnishing,is_boost
0,2bdrm Block of Flats in Ago Palace for sale,"Isolo, Ago Palace",Ago Palace,Isolo,270000000,2,2,600,Unfurnished,diamond
1,5bdrm Duplex in Ikota for sale,"Lekki, Ikota",Ikota,Lekki,650000000,5,5,500,Unfurnished,diamond
2,5bdrm House in Ikota for sale,"Lekki, Ikota",Ikota,Lekki,600000000,5,5,550,Unfurnished,diamond
3,3bdrm Apartment in Ago Palace for sale,"Isolo, Ago Palace",Ago Palace,Isolo,90000000,3,3,620,Unfurnished,diamond
4,6bdrm Duplex in GRA Phase 1 for sale,"Magodo, GRA Phase 1",GRA Phase 1,Magodo,450000000,6,6,300,Unfurnished,diamond
5,4bdrm Duplex in Ago Palace for sale,"Isolo, Ago Palace",Ago Palace,Isolo,260000000,4,4,220,Unfurnished,diamond
6,4bdrm Bungalow in Gberigbe for sale,"Ikorodu, Gberigbe",Gberigbe,Ikorodu,22000000,4,4,638,Unfurnished,diamond
7,5bdrm Duplex in Abraham Adesanya Estate for sale,"Ajah, Abraham Adesanya Estate",Abraham Adesanya Estate,Ajah,260000000,5,5,500,Unfurnished,diamond
8,4bdrm Duplex in Chevron for sale,"Lekki, Chevron",Chevron,Lekki,200000000,4,4,1000,Unfurnished,diamond
9,5bdrm Duplex in Banana Island for sale,"Ikoyi, Banana Island",Banana Island,Ikoyi,7000000000,5,5,600,Unfurnished,diamond


In [124]:
#exclude an outliers in dataframe

Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)
Q1
Q3
IQR = Q3 -Q1
IQR

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = ( df['price'] < lower_bound) | ( df['price'] > upper_bound)

print(outliers.sum())

new_df = df[
    (df['price'] > lower_bound) & ( df['price'] < upper_bound)
] 

new_df

2


,title,region,region_name,region_parent_name,price,bedrooms,bathrooms,property_size,furnishing,is_boost
0,2bdrm Block of Flats in Ago Palace for sale,"Isolo, Ago Palace",Ago Palace,Isolo,270000000,2,2,600,Unfurnished,diamond
1,5bdrm Duplex in Ikota for sale,"Lekki, Ikota",Ikota,Lekki,650000000,5,5,500,Unfurnished,diamond
2,5bdrm House in Ikota for sale,"Lekki, Ikota",Ikota,Lekki,600000000,5,5,550,Unfurnished,diamond
3,3bdrm Apartment in Ago Palace for sale,"Isolo, Ago Palace",Ago Palace,Isolo,90000000,3,3,620,Unfurnished,diamond
4,6bdrm Duplex in GRA Phase 1 for sale,"Magodo, GRA Phase 1",GRA Phase 1,Magodo,450000000,6,6,300,Unfurnished,diamond
5,4bdrm Duplex in Ago Palace for sale,"Isolo, Ago Palace",Ago Palace,Isolo,260000000,4,4,220,Unfurnished,diamond
6,4bdrm Bungalow in Gberigbe for sale,"Ikorodu, Gberigbe",Gberigbe,Ikorodu,22000000,4,4,638,Unfurnished,diamond
7,5bdrm Duplex in Abraham Adesanya Estate for sale,"Ajah, Abraham Adesanya Estate",Abraham Adesanya Estate,Ajah,260000000,5,5,500,Unfurnished,diamond
8,4bdrm Duplex in Chevron for sale,"Lekki, Chevron",Chevron,Lekki,200000000,4,4,1000,Unfurnished,diamond
10,2bdrm Apartment in Lekki Phase 1 for sale,"Lekki, Lekki Phase 1",Lekki Phase 1,Lekki,310000000,2,2,900,Unfurnished,diamond


In [125]:
new_df["furnishing"] = (
    new_df["furnishing"]
    .str.lower()
    .replace({
        "semi furnished": "Semi-Furnished",
        "furnished": "Furnished",
        "unfurnished": "Unfurnished"
    })
)

new_df["furnishing"] = new_df["furnishing"].fillna("Unknown")
new_df

,title,region,region_name,region_parent_name,price,bedrooms,bathrooms,property_size,furnishing,is_boost
0,2bdrm Block of Flats in Ago Palace for sale,"Isolo, Ago Palace",Ago Palace,Isolo,270000000,2,2,600,Unfurnished,diamond
1,5bdrm Duplex in Ikota for sale,"Lekki, Ikota",Ikota,Lekki,650000000,5,5,500,Unfurnished,diamond
2,5bdrm House in Ikota for sale,"Lekki, Ikota",Ikota,Lekki,600000000,5,5,550,Unfurnished,diamond
3,3bdrm Apartment in Ago Palace for sale,"Isolo, Ago Palace",Ago Palace,Isolo,90000000,3,3,620,Unfurnished,diamond
4,6bdrm Duplex in GRA Phase 1 for sale,"Magodo, GRA Phase 1",GRA Phase 1,Magodo,450000000,6,6,300,Unfurnished,diamond
5,4bdrm Duplex in Ago Palace for sale,"Isolo, Ago Palace",Ago Palace,Isolo,260000000,4,4,220,Unfurnished,diamond
6,4bdrm Bungalow in Gberigbe for sale,"Ikorodu, Gberigbe",Gberigbe,Ikorodu,22000000,4,4,638,Unfurnished,diamond
7,5bdrm Duplex in Abraham Adesanya Estate for sale,"Ajah, Abraham Adesanya Estate",Abraham Adesanya Estate,Ajah,260000000,5,5,500,Unfurnished,diamond
8,4bdrm Duplex in Chevron for sale,"Lekki, Chevron",Chevron,Lekki,200000000,4,4,1000,Unfurnished,diamond
10,2bdrm Apartment in Lekki Phase 1 for sale,"Lekki, Lekki Phase 1",Lekki Phase 1,Lekki,310000000,2,2,900,Unfurnished,diamond


In [126]:
new_df.rename(columns={
    "title" : "Title",
    "price" : "Price",
    "bedrooms" : "Bedrooms",
    "bathrooms" : "Bathrooms",
    "property_size" : "Property Size",
    "furnishing" : "Furnishing",
    "is_boost" : "Is Boost",
    "region" : "Region",
    "region_name": "Region Name",
    "region_parent_name": "Region Parent Name"
    }, inplace=True)
new_df

,Title,Region,Region Name,Region Parent Name,Price,Bedrooms,Bathrooms,Property Size,Furnishing,Is Boost
0,2bdrm Block of Flats in Ago Palace for sale,"Isolo, Ago Palace",Ago Palace,Isolo,270000000,2,2,600,Unfurnished,diamond
1,5bdrm Duplex in Ikota for sale,"Lekki, Ikota",Ikota,Lekki,650000000,5,5,500,Unfurnished,diamond
2,5bdrm House in Ikota for sale,"Lekki, Ikota",Ikota,Lekki,600000000,5,5,550,Unfurnished,diamond
3,3bdrm Apartment in Ago Palace for sale,"Isolo, Ago Palace",Ago Palace,Isolo,90000000,3,3,620,Unfurnished,diamond
4,6bdrm Duplex in GRA Phase 1 for sale,"Magodo, GRA Phase 1",GRA Phase 1,Magodo,450000000,6,6,300,Unfurnished,diamond
5,4bdrm Duplex in Ago Palace for sale,"Isolo, Ago Palace",Ago Palace,Isolo,260000000,4,4,220,Unfurnished,diamond
6,4bdrm Bungalow in Gberigbe for sale,"Ikorodu, Gberigbe",Gberigbe,Ikorodu,22000000,4,4,638,Unfurnished,diamond
7,5bdrm Duplex in Abraham Adesanya Estate for sale,"Ajah, Abraham Adesanya Estate",Abraham Adesanya Estate,Ajah,260000000,5,5,500,Unfurnished,diamond
8,4bdrm Duplex in Chevron for sale,"Lekki, Chevron",Chevron,Lekki,200000000,4,4,1000,Unfurnished,diamond
10,2bdrm Apartment in Lekki Phase 1 for sale,"Lekki, Lekki Phase 1",Lekki Phase 1,Lekki,310000000,2,2,900,Unfurnished,diamond


In [127]:
def extract_size(x):

    if pd.isna(x):
        return None

    nums = re.findall(r"\d+", str(x))

    if nums:
        return float(nums[0])

    return None

new_df["Property Size"] = new_df["Property Size"].apply(extract_size)
new_df

,Title,Region,Region Name,Region Parent Name,Price,Bedrooms,Bathrooms,Property Size,Furnishing,Is Boost
0,2bdrm Block of Flats in Ago Palace for sale,"Isolo, Ago Palace",Ago Palace,Isolo,270000000,2,2,600.0,Unfurnished,diamond
1,5bdrm Duplex in Ikota for sale,"Lekki, Ikota",Ikota,Lekki,650000000,5,5,500.0,Unfurnished,diamond
2,5bdrm House in Ikota for sale,"Lekki, Ikota",Ikota,Lekki,600000000,5,5,550.0,Unfurnished,diamond
3,3bdrm Apartment in Ago Palace for sale,"Isolo, Ago Palace",Ago Palace,Isolo,90000000,3,3,620.0,Unfurnished,diamond
4,6bdrm Duplex in GRA Phase 1 for sale,"Magodo, GRA Phase 1",GRA Phase 1,Magodo,450000000,6,6,300.0,Unfurnished,diamond
5,4bdrm Duplex in Ago Palace for sale,"Isolo, Ago Palace",Ago Palace,Isolo,260000000,4,4,220.0,Unfurnished,diamond
6,4bdrm Bungalow in Gberigbe for sale,"Ikorodu, Gberigbe",Gberigbe,Ikorodu,22000000,4,4,638.0,Unfurnished,diamond
7,5bdrm Duplex in Abraham Adesanya Estate for sale,"Ajah, Abraham Adesanya Estate",Abraham Adesanya Estate,Ajah,260000000,5,5,500.0,Unfurnished,diamond
8,4bdrm Duplex in Chevron for sale,"Lekki, Chevron",Chevron,Lekki,200000000,4,4,1000.0,Unfurnished,diamond
10,2bdrm Apartment in Lekki Phase 1 for sale,"Lekki, Lekki Phase 1",Lekki Phase 1,Lekki,310000000,2,2,900.0,Unfurnished,diamond


In [128]:
new_df.drop_duplicates(inplace=True)
new_df

,Title,Region,Region Name,Region Parent Name,Price,Bedrooms,Bathrooms,Property Size,Furnishing,Is Boost
0,2bdrm Block of Flats in Ago Palace for sale,"Isolo, Ago Palace",Ago Palace,Isolo,270000000,2,2,600.0,Unfurnished,diamond
1,5bdrm Duplex in Ikota for sale,"Lekki, Ikota",Ikota,Lekki,650000000,5,5,500.0,Unfurnished,diamond
2,5bdrm House in Ikota for sale,"Lekki, Ikota",Ikota,Lekki,600000000,5,5,550.0,Unfurnished,diamond
3,3bdrm Apartment in Ago Palace for sale,"Isolo, Ago Palace",Ago Palace,Isolo,90000000,3,3,620.0,Unfurnished,diamond
4,6bdrm Duplex in GRA Phase 1 for sale,"Magodo, GRA Phase 1",GRA Phase 1,Magodo,450000000,6,6,300.0,Unfurnished,diamond
5,4bdrm Duplex in Ago Palace for sale,"Isolo, Ago Palace",Ago Palace,Isolo,260000000,4,4,220.0,Unfurnished,diamond
6,4bdrm Bungalow in Gberigbe for sale,"Ikorodu, Gberigbe",Gberigbe,Ikorodu,22000000,4,4,638.0,Unfurnished,diamond
7,5bdrm Duplex in Abraham Adesanya Estate for sale,"Ajah, Abraham Adesanya Estate",Abraham Adesanya Estate,Ajah,260000000,5,5,500.0,Unfurnished,diamond
8,4bdrm Duplex in Chevron for sale,"Lekki, Chevron",Chevron,Lekki,200000000,4,4,1000.0,Unfurnished,diamond
10,2bdrm Apartment in Lekki Phase 1 for sale,"Lekki, Lekki Phase 1",Lekki Phase 1,Lekki,310000000,2,2,900.0,Unfurnished,diamond


In [129]:
new_df.to_csv("data/jiji_housing_cleaned.csv", index=False)

In [130]:
new_df["Price"].mean()

np.float64(297611111.1111111)

In [131]:
state_price = (
    new_df.groupby("Region Name")["Price"]
    .mean()
    .sort_values(ascending=False)
)
print("Highest Mean Price State:")
print(state_price.head())

print("\nLowest Mean Price State:")
print(state_price.tail())

Highest Mean Price State:
Region Name
Victoria Island Extension    680000000.0
Ikota                        625000000.0
GRA Phase 1                  450000000.0
Basorun                      420000000.0
Mabushi                      380000000.0
Name: Price, dtype: float64

Lowest Mean Price State:
Region Name
Chevron           200000000.0
FHA                95000000.0
Gaduwa             95000000.0
Oke Ira / Ajah     80000000.0
Gberigbe           22000000.0
Name: Price, dtype: float64


In [132]:
new_df["Property Size"].value_counts()

Property Size
500.0     4
300.0     2
600.0     1
550.0     1
620.0     1
220.0     1
638.0     1
1000.0    1
900.0     1
850.0     1
2200.0    1
450.0     1
150.0     1
250.0     1
Name: count, dtype: int64

In [133]:
region_counts = new_df["Region Name"].value_counts()

print(region_counts.head(10))

Region Name
Ago Palace                   4
Ikota                        2
GRA Phase 1                  1
Gberigbe                     1
Abraham Adesanya Estate      1
Chevron                      1
Lekki Phase 1                1
Victoria Island Extension    1
Life Camp                    1
Oke Ira / Ajah               1
Name: count, dtype: int64


In [134]:
furnishing_price = (
    new_df.groupby("Furnishing")["Price"]
    .mean()
    .sort_values(ascending=False)
)

print(furnishing_price)

Furnishing
semi-furnished    4.200000e+08
Unfurnished       2.915714e+08
Furnished         2.850000e+08
Name: Price, dtype: float64


In [135]:
boost_price = (
    new_df.groupby("Is Boost")["Price"]
    .mean()
)

print(boost_price)

Is Boost
False      3.433333e+08
diamond    2.860000e+08
premium    2.983333e+08
Name: Price, dtype: float64


In [136]:
#Correlation Analysis (Bonus)
corr = new_df[
    [
        "Price",
        "Property Size",
        "Bedrooms",
        "Bathrooms"
    ]
].corr()

corr

,Price,Property Size,Bedrooms,Bathrooms
Price,1.000000,0.391470,0.484834,0.447805
Property Size,0.391470,1.000000,-0.117083,-0.126744
Bedrooms,0.484834,-0.117083,1.000000,0.983801
Bathrooms,0.447805,-0.126744,0.983801,1.000000


In [137]:
new_df.to_csv("data/cleaned_nigeria_housing_dataset.csv", index=False)